In [0]:
from pyspark.sql import functions as F, Window as W

cat = "capstone6_telecom.telecom"

EXTREME_LATENCY_MS = 500                      # TC02: exactly 35 rows are above this
EVENT_STATUS = ["OPEN", "FAILED", "PENDING", "COMPLETED"]
CDC_STATUS   = EVENT_STATUS + ["CANCELLED", "REVERSED"]
SEC_TYPES    = ["CHECK", "ALERT", "REVIEW"]
OUTAGE_STATUS = "FAILED"                      # a reading counts as an outage when its CURRENT status is this

In [0]:
def failed_rules(*pairs):
    """pairs = (condition_meaning_the_rule_FAILED, rule_id). Returns an array of failed rule ids."""
    return F.filter(F.array(*[F.when(c, F.lit(r)) for c, r in pairs]), lambda x: x.isNotNull())

def not_in(col, values):
    """True when the value is null or not in the list."""
    return F.coalesce(~F.col(col).isin(values), F.lit(True))

def rejected_rows(df, record_type, key_col, raw_cols):
    """df needs: rule_id, action, source_file, ingest_ts, key_col and raw_cols."""
    rules = spark.table(f"{cat}.silver_dq_rules").select("rule_id", F.col("description").alias("reason"))
    return (df.join(F.broadcast(rules), "rule_id", "left")
              .select(F.lit(record_type).alias("record_type"),
                      F.col(key_col).cast("string").alias("record_key"),
                      "rule_id", "action", "reason", "source_file",
                      F.col("ingest_ts").alias("bronze_ingest_ts"),
                      F.to_json(F.struct(*raw_cols)).alias("raw_record"),
                      F.current_timestamp().alias("rejected_at")))

def write_rejected(df, record_type):
    """Idempotent: replaces only this record_type's rows, so re-runs never duplicate."""
    t = f"{cat}.silver_rejected_records"
    if spark.catalog.tableExists(t):
        (df.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"record_type = '{record_type}'").saveAsTable(t))
    else:
        df.write.format("delta").saveAsTable(t)

### Function: build Silver events (just run this cell)

In [0]:
def build_silver_events(b, regions, towers):
    """Bronze events -> (clean, rejected). Keeps raw values in Bronze; types and flags here."""
    if "source_schema_version" not in b.columns:
        b = b.withColumn("source_schema_version", F.lit(None).cast("string"))
    raw_cols = b.columns

    t = (b
         .withColumn("event_ts", F.expr("try_to_timestamp(event_timestamp)"))
         .withColumn("source_ingest_ts", F.expr("try_to_timestamp(ingest_timestamp)"))
         .withColumn("latency_n", F.expr("try_cast(latency_ms AS DOUBLE)"))
         .withColumn("loss_n", F.expr("try_cast(packet_loss_pct AS DOUBLE)"))
         .withColumn("status_c", F.upper(F.trim(F.col("network_status"))))
         .withColumn("type_c", F.expr("nullif(upper(trim(network_type)), '')")))

    # Deterministic dedup: newest Bronze load wins, then highest file name.
    w = W.partitionBy("measurement_id").orderBy(F.col("ingest_ts").desc(), F.col("source_file").desc())
    t = (t.withColumn("rn", F.row_number().over(w))
          .withColumn("copies", F.count("*").over(W.partitionBy("measurement_id")))
          .withColumn("is_dup", F.col("measurement_id").isNotNull() & (F.col("rn") > 1)))

    t = (t.join(F.broadcast(regions.select("region_id").distinct().withColumn("_reg_ok", F.lit(True))), "region_id", "left")
          .join(F.broadcast(towers.select("tower_id").distinct().withColumn("_twr_ok", F.lit(True))), "tower_id", "left"))

    t = t.withColumn("failed", failed_rules(
        (F.col("measurement_id").isNull() | (F.trim(F.col("measurement_id")) == ""), "DQ01"),
        (F.col("event_ts").isNull(), "DQ02"),
        (F.col("latency_n").isNull() | (F.col("latency_n") < 0), "DQ03"),
        (F.col("loss_n").isNull() | (F.col("loss_n") < 0) | (F.col("loss_n") > 100), "DQ04"),
        (not_in("status_c", EVENT_STATUS), "DQ05"),
        (F.col("_reg_ok").isNull(), "DQ06"),
        (F.col("_twr_ok").isNull(), "DQ07")))

    clean = (t.filter(~F.col("is_dup") & (F.size("failed") == 0))
              .select("measurement_id", "event_ts", "source_ingest_ts", "region_id", "tower_id",
                      F.coalesce("type_c", F.lit("UNKNOWN")).alias("network_type"),
                      F.col("type_c").isNull().alias("network_type_missing"),          # DQ09 flag
                      F.col("status_c").alias("network_status"), "unit",
                      F.col("latency_n").alias("latency_ms"),
                      F.col("loss_n").alias("packet_loss_pct"),
                      (F.col("latency_n") > EXTREME_LATENCY_MS).alias("latency_outlier_flag"),  # DQ10 flag
                      "operator_id", "source_system", "source_schema_version",
                      (F.col("copies") - 1).alias("duplicate_copies_dropped"),
                      "source_file", F.col("ingest_ts").alias("bronze_ingest_ts")))

    rej_fail = (t.filter(~F.col("is_dup") & (F.size("failed") > 0))
                 .withColumn("rule_id", F.explode("failed")).withColumn("action", F.lit("REJECTED")))
    rej_dup = (t.filter(F.col("is_dup"))
                .withColumn("rule_id", F.lit("DQ08")).withColumn("action", F.lit("DEDUPED")))
    rejected = rejected_rows(rej_fail.unionByName(rej_dup), "EVENT", "measurement_id", raw_cols)
    return clean, rejected

### Functions: validate and apply status CDC (just run this cell)

In [0]:
def validate_cdc(raw, events):
    """Bronze CDC rows -> (valid, rejected). events needs measurement_id and event_ts."""
    raw_cols = raw.columns
    c = (raw
         .withColumn("change_ts", F.expr("try_to_timestamp(change_timestamp)"))
         .withColumn("seq", F.expr("try_cast(source_sequence AS BIGINT)"))
         .withColumn("prev_c", F.upper(F.trim(F.col("previous_status"))))
         .withColumn("new_c", F.upper(F.trim(F.col("new_status"))))
         .join(events.select("measurement_id", F.col("event_ts").alias("evt_ts")), "measurement_id", "left"))

    w = W.partitionBy("measurement_id", "seq").orderBy(F.col("ingest_ts").desc(), F.col("source_file").desc())
    c = (c.withColumn("rn", F.row_number().over(w))
          .withColumn("is_dup", F.col("seq").isNotNull() & (F.col("rn") > 1))
          .withColumn("failed", failed_rules(
              (F.col("evt_ts").isNull(), "DQ11"),                                   # orphan change
              (not_in("new_c", CDC_STATUS), "DQ12"),                                # unknown status
              (F.col("change_ts").isNull() | (F.col("change_ts") < F.col("evt_ts")), "DQ13"),  # bad / impossible time
              (F.col("seq").isNull(), "DQ14"))))                                    # no sequence

    valid = (c.filter(~F.col("is_dup") & (F.size("failed") == 0))
              .select("measurement_id", F.col("seq").alias("source_sequence"), "change_ts",
                      F.col("prev_c").alias("previous_status"), F.col("new_c").alias("new_status"),
                      "change_reason", "source_system", "source_file",
                      F.col("ingest_ts").alias("bronze_ingest_ts")))

    rej_fail = (c.filter(~F.col("is_dup") & (F.size("failed") > 0))
                 .withColumn("rule_id", F.explode("failed")).withColumn("action", F.lit("REJECTED")))
    rej_dup = c.filter(F.col("is_dup")).withColumn("rule_id", F.lit("DQ15")).withColumn("action", F.lit("DEDUPED"))
    rejected = rejected_rows(rej_fail.unionByName(rej_dup), "CDC", "measurement_id", raw_cols)
    return valid, rejected

def create_cdc_tables(history, current):
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {history} (
        measurement_id STRING, source_sequence BIGINT, change_ts TIMESTAMP,
        previous_status STRING, new_status STRING, status_changed BOOLEAN,
        change_reason STRING, source_system STRING, source_file STRING,
        bronze_ingest_ts TIMESTAMP, loaded_at TIMESTAMP)""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {current} (
        measurement_id STRING, region_id STRING, tower_id STRING, event_ts TIMESTAMP,
        initial_status STRING, current_status STRING, last_sequence BIGINT,
        last_change_ts TIMESTAMP, last_change_reason STRING, updated_at TIMESTAMP)""")

def apply_cdc(valid, events, history, current):
    """events needs: measurement_id, region_id, tower_id, event_ts, network_status."""
    events.select("measurement_id", "region_id", "tower_id", "event_ts", "network_status") \
          .createOrReplaceTempView("_cdc_events")
    valid.createOrReplaceTempView("_cdc_valid")

    # 1) seed: every event gets a current-state row (only new ones are inserted)
    spark.sql(f"""
      MERGE INTO {current} t USING _cdc_events s ON t.measurement_id = s.measurement_id
      WHEN NOT MATCHED THEN INSERT (measurement_id, region_id, tower_id, event_ts, initial_status,
           current_status, last_sequence, last_change_ts, last_change_reason, updated_at)
      VALUES (s.measurement_id, s.region_id, s.tower_id, s.event_ts, s.network_status,
           s.network_status, 0, s.event_ts, 'INITIAL_EVENT', current_timestamp())""")

    # 2) history: every valid change, once. Replays and late rows are kept, never duplicated.
    spark.sql(f"""
      MERGE INTO {history} t
      USING (SELECT measurement_id, source_sequence, change_ts, previous_status, new_status,
                    previous_status <> new_status AS status_changed, change_reason, source_system,
                    source_file, bronze_ingest_ts, current_timestamp() AS loaded_at FROM _cdc_valid) s
      ON t.measurement_id = s.measurement_id AND t.source_sequence = s.source_sequence
      WHEN NOT MATCHED THEN INSERT (measurement_id, source_sequence, change_ts, previous_status, new_status,
           status_changed, change_reason, source_system, source_file, bronze_ingest_ts, loaded_at)
      VALUES (s.measurement_id, s.source_sequence, s.change_ts, s.previous_status, s.new_status,
           s.status_changed, s.change_reason, s.source_system, s.source_file, s.bronze_ingest_ts, s.loaded_at)""")

    # 3) current: only a HIGHER source_sequence may overwrite, so late rows never beat newer status
    spark.sql(f"""
      MERGE INTO {current} t
      USING (SELECT measurement_id, source_sequence, change_ts, new_status, change_reason FROM (
               SELECT *, ROW_NUMBER() OVER (PARTITION BY measurement_id ORDER BY source_sequence DESC) AS rn
               FROM _cdc_valid) WHERE rn = 1) s
      ON t.measurement_id = s.measurement_id
      WHEN MATCHED AND s.source_sequence > t.last_sequence THEN UPDATE SET
           t.current_status = s.new_status, t.last_sequence = s.source_sequence,
           t.last_change_ts = s.change_ts, t.last_change_reason = s.change_reason,
           t.updated_at = current_timestamp()""")

### Function: build Silver secondary events (just run this cell)

In [0]:
def build_silver_secondary(b, events, regions):
    """Bronze secondary events -> (clean, rejected)."""
    raw_cols = b.columns
    s = (b
         .withColumn("event_ts", F.expr("try_to_timestamp(event_timestamp)"))
         .withColumn("value_n", F.expr("try_cast(event_value AS DOUBLE)"))
         .withColumn("type_c", F.upper(F.trim(F.col("event_type"))))
         .join(F.broadcast(events.select("measurement_id").distinct().withColumn("_evt_ok", F.lit(True))), "measurement_id", "left")
         .join(F.broadcast(regions.select(F.col("region_id").alias("related_entity_id")).distinct().withColumn("_reg_ok", F.lit(True))),
               "related_entity_id", "left"))

    w = W.partitionBy("event_id").orderBy(F.col("ingest_ts").desc(), F.col("source_file").desc())
    s = (s.withColumn("rn", F.row_number().over(w))
          .withColumn("is_dup", F.col("event_id").isNotNull() & (F.col("rn") > 1))
          .withColumn("failed", failed_rules(
              (F.col("event_id").isNull(), "DQ16"),
              (F.col("_evt_ok").isNull(), "DQ17"),
              (not_in("type_c", SEC_TYPES), "DQ18"),
              (F.col("value_n").isNull(), "DQ19"),
              (F.col("event_ts").isNull(), "DQ20"),
              (F.col("_reg_ok").isNull(), "DQ21"))))

    clean = (s.filter(~F.col("is_dup") & (F.size("failed") == 0))
              .select("event_id", "measurement_id", "event_ts",
                      F.col("related_entity_id").alias("related_region_id"),
                      F.col("type_c").alias("event_type"), F.col("value_n").alias("event_value"),
                      "source_system", "source_file", F.col("ingest_ts").alias("bronze_ingest_ts")))
    rej_fail = (s.filter(~F.col("is_dup") & (F.size("failed") > 0))
                 .withColumn("rule_id", F.explode("failed")).withColumn("action", F.lit("REJECTED")))
    rej_dup = s.filter(F.col("is_dup")).withColumn("rule_id", F.lit("DQ16")).withColumn("action", F.lit("DEDUPED"))
    rejected = rejected_rows(rej_fail.unionByName(rej_dup), "SECONDARY", "event_id", raw_cols)
    return clean, rejected

## A. CDC tests (TC04)

In [0]:
th, tc = f"{cat}.test_cdc_history", f"{cat}.test_cdc_current"
spark.sql(f"DROP TABLE IF EXISTS {th}")
spark.sql(f"DROP TABLE IF EXISTS {tc}")
create_cdc_tables(th, tc)

t_events = (spark.createDataFrame(
    [("T-1", "R-1", "W-1", "2026-07-01T00:00:00", "OPEN"),
     ("T-2", "R-1", "W-1", "2026-07-01T00:05:00", "COMPLETED"),
     ("T-3", "R-2", "W-2", "2026-07-01T00:10:00", "FAILED")],
    "measurement_id string, region_id string, tower_id string, event_timestamp string, network_status string")
    .withColumn("event_ts", F.expr("try_to_timestamp(event_timestamp)")))

def cdc_rows(rows):
    return (spark.createDataFrame(rows,
        "measurement_id string, change_timestamp string, previous_status string, new_status string, "
        "change_reason string, source_sequence string, source_system string")
        .withColumn("source_file", F.lit("test.csv")).withColumn("ingest_ts", F.current_timestamp()))

def run(rows):
    valid, rejected = validate_cdc(cdc_rows(rows), t_events)
    apply_cdc(valid, t_events, th, tc)
    return rejected

def cur(mid):
    return spark.sql(f"SELECT current_status FROM {tc} WHERE measurement_id='{mid}'").collect()[0][0]

def hist(mid):
    return spark.sql(f"SELECT COUNT(*) FROM {th} WHERE measurement_id='{mid}'").collect()[0][0]

In [0]:
# 1. Ordered later revisions: latest wins, history keeps both
run([("T-1", "2026-08-10T01:00:00", "OPEN", "COMPLETED", "SYSTEM_REVIEW", "10", "S")])
run([("T-1", "2026-08-10T02:00:00", "COMPLETED", "REVERSED", "CUSTOMER_ACTION", "11", "S")])
assert cur("T-1") == "REVERSED" and hist("T-1") == 2, "later revision test failed"

# 2. Late / out-of-order: older sequence arrives after a newer one -> kept in history, does NOT overwrite
run([("T-1", "2026-08-10T00:30:00", "OPEN", "FAILED", "OPERATIONS_UPDATE", "9", "S")])
assert cur("T-1") == "REVERSED" and hist("T-1") == 3, "late event overwrote newer status"

# 3. Replay: same rows again -> nothing changes
run([("T-1", "2026-08-10T01:00:00", "OPEN", "COMPLETED", "SYSTEM_REVIEW", "10", "S"),
     ("T-1", "2026-08-10T02:00:00", "COMPLETED", "REVERSED", "CUSTOMER_ACTION", "11", "S")])
assert cur("T-1") == "REVERSED" and hist("T-1") == 3, "replay created duplicates"

# 4. Unchanged status (COMPLETED -> COMPLETED): valid, recorded, flagged as not changed
run([("T-2", "2026-08-10T03:00:00", "COMPLETED", "COMPLETED", "SYSTEM_REVIEW", "12", "S")])
assert cur("T-2") == "COMPLETED"
assert spark.sql(f"SELECT status_changed FROM {th} WHERE measurement_id='T-2'").collect()[0][0] is False

# 5. Several changes for one key inside one batch -> highest sequence wins
run([("T-3", "2026-08-10T04:00:00", "FAILED", "CANCELLED", "SYSTEM_REVIEW", "20", "S"),
     ("T-3", "2026-08-10T05:00:00", "CANCELLED", "COMPLETED", "SYSTEM_REVIEW", "21", "S")])
assert cur("T-3") == "COMPLETED" and hist("T-3") == 2
print("CDC tests passed")

## B. Malformed CDC input is rejected with a rule id (TC05)

In [0]:
before = spark.table(th).count()
rejected = run([
    ("T-99", "2026-08-10T06:00:00", "OPEN", "COMPLETED", "SYSTEM_REVIEW", "30", "S"),   # DQ11 orphan
    ("T-1",  "2026-08-10T06:00:00", "OPEN", "NOT_A_STATUS", "SYSTEM_REVIEW", "31", "S"), # DQ12 bad status
    ("T-1",  "2026-06-01T00:00:00", "OPEN", "COMPLETED", "SYSTEM_REVIEW", "32", "S"),   # DQ13 before event
    ("T-1",  "not-a-date",          "OPEN", "COMPLETED", "SYSTEM_REVIEW", "33", "S"),   # DQ13 unparseable
    ("T-1",  "2026-08-10T06:00:00", "OPEN", "COMPLETED", "SYSTEM_REVIEW", "abc", "S"),  # DQ14 bad sequence
    ("T-2",  "2026-08-10T07:00:00", "COMPLETED", "REVERSED", "SYSTEM_REVIEW", "40", "S"),
    ("T-2",  "2026-08-10T07:00:00", "COMPLETED", "REVERSED", "SYSTEM_REVIEW", "40", "S"),  # DQ15 duplicate in batch
])
got = sorted({r["rule_id"] for r in rejected.collect()})
print(got)
assert got == ["DQ11", "DQ12", "DQ13", "DQ14", "DQ15"], got
assert spark.table(th).count() == before + 1, "only the one valid row (seq 40) may be added"
assert cur("T-2") == "REVERSED"
print("Malformed CDC tests passed")

## C. Event rules (duplicates, invalid rows, flags)

In [0]:
b_schema = ("measurement_id string, event_timestamp string, ingest_timestamp string, region_id string, tower_id string, "
            "network_type string, network_status string, unit string, latency_ms string, packet_loss_pct string, "
            "operator_id string, source_system string, source_file string")
rows = [
 ("E1", "2026-07-01T00:00:00", "2026-07-01T00:05:00", "R-1", "W-1", "5G", "OPEN",   "METRIC", "30.5", "1.0", "O-1", "S", "f1.csv"),
 ("E1", "2026-07-01T00:00:00", "2026-07-01T00:05:00", "R-1", "W-1", "5G", "OPEN",   "METRIC", "30.5", "1.0", "O-1", "S", "f2.csv"),  # exact duplicate
 ("E2", "2026-07-01T00:01:00", "2026-07-01T00:06:00", "R-1", "W-1", None, "FAILED", "METRIC", "900",  "2.0", "O-1", "S", "f1.csv"),  # missing type + outlier -> kept
 ("E3", "bad-date",            "2026-07-01T00:06:00", "R-1", "W-1", "4G", "OPEN",   "METRIC", "20",   "2.0", "O-1", "S", "f1.csv"),  # DQ02
 ("E4", "2026-07-01T00:02:00", "2026-07-01T00:07:00", "R-1", "W-1", "4G", "OPEN",   "METRIC", "-5",   "2.0", "O-1", "S", "f1.csv"),  # DQ03
 ("E5", "2026-07-01T00:03:00", "2026-07-01T00:08:00", "R-1", "W-1", "4G", "OPEN",   "METRIC", "20",   "150", "O-1", "S", "f1.csv"),  # DQ04
 ("E6", "2026-07-01T00:04:00", "2026-07-01T00:09:00", "R-1", "W-1", "4G", "WEIRD",  "METRIC", "20",   "2.0", "O-1", "S", "f1.csv"),  # DQ05
 ("E7", "2026-07-01T00:05:00", "2026-07-01T00:10:00", "R-X", "W-1", "4G", "OPEN",   "METRIC", "20",   "2.0", "O-1", "S", "f1.csv"),  # DQ06
 ("E8", "2026-07-01T00:06:00", "2026-07-01T00:11:00", "R-1", "W-X", "4G", "OPEN",   "METRIC", "20",   "2.0", "O-1", "S", "f1.csv"),  # DQ07
 (None, "2026-07-01T00:07:00", "2026-07-01T00:12:00", "R-1", "W-1", "4G", "OPEN",   "METRIC", "20",   "2.0", "O-1", "S", "f1.csv"),  # DQ01
]
tb = spark.createDataFrame(rows, b_schema).withColumn("ingest_ts", F.current_timestamp())
regions = spark.createDataFrame([("R-1",)], "region_id string")
towers  = spark.createDataFrame([("W-1",)], "tower_id string")

clean, rejected = build_silver_events(tb, regions, towers)
kept = {r["measurement_id"]: r for r in clean.collect()}
assert sorted(kept) == ["E1", "E2"], sorted(kept)
assert kept["E2"]["network_type"] == "UNKNOWN" and kept["E2"]["network_type_missing"] is True
assert kept["E2"]["latency_outlier_flag"] is True, "outlier must be flagged, not removed"
assert kept["E1"]["duplicate_copies_dropped"] == 1 and kept["E1"]["source_file"] == "f2.csv", "newest copy should win"
rules_hit = sorted({r["rule_id"] for r in rejected.collect()})
assert rules_hit == ["DQ01", "DQ02", "DQ03", "DQ04", "DQ05", "DQ06", "DQ07", "DQ08"], rules_hit
print("Event rule tests passed")

## Clean up test tables

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {th}")
spark.sql(f"DROP TABLE IF EXISTS {tc}")
print("All Silver unit tests passed")